# CarsiInduce 虚拟筛选教程

**论文**: CarsiInduce -- Ligand-induced protein conformational change prediction via cross-attention  

**核心创新**: CarsiInduce 关注 **诱导契合对接 (induced-fit docking)** 问题。与传统半柔性对接只预测配体位姿不同，CarsiInduce 学习"配体如何诱导蛋白质从 apo (未结合) 构象变为 holo (结合) 构象"。模型利用 **残基级交叉注意力 (cross-attention)** 机制，让配体原子信息作为 Key/Value 引导蛋白质残基 (Query) 的构象校正。

**方法概述**:
1. 从蛋白质口袋中提取残基级表示 (CA 坐标 + 残基类型 one-hot)
2. 对真实 holo 构象施加随机扰动 (平移 + 旋转)，模拟 apo 构象
3. 利用配体原子特征作为 Key/Value，扰动残基特征作为 Query，通过交叉注意力机制预测每个残基的校正 (平移 + 旋转)
4. 将校正应用于扰动后的残基坐标，恢复接近 holo 的构象

**学习目标**:
- 理解诱导契合对接 (induced-fit docking) 的核心思想
- 掌握残基级蛋白质构象表示 (CA 坐标 + 残基类型 one-hot)
- 理解交叉注意力机制如何实现配体对蛋白质构象的引导
- 学习 Rodrigues 旋转公式在刚体变换中的应用
- 使用 RMSD 评估蛋白质构象预测质量

In [ ]:
# ============================================================
# 路径设置与依赖导入
# ============================================================
import os
import warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit import RDLogger
from Bio.PDB import PDBParser
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

# 抑制 RDKit 的冗余警告信息
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')


def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    from pathlib import Path
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'数据目录:   {DATA_DIR}')

# 版本信息
version_info = pd.DataFrame({
    '依赖库': ['torch', 'rdkit', 'numpy', 'biopython', 'matplotlib', 'pandas'],
    '版本': [
        torch.__version__,
        Chem.rdBase.rdkitVersion,
        np.__version__,
        __import__('Bio').__version__,
        __import__('matplotlib').__version__,
        pd.__version__,
    ]
})
display(version_info)

## 1. 超参数设置

CarsiInduce 的关键超参数包括：
- **残基/配体特征维度**: 残基用 21 维 one-hot (20 种标准氨基酸 + 1 other)，配体原子用 10 维特征
- **交叉注意力层数**: 控制配体信息传递给蛋白质残基的深度
- **扰动参数**: 控制模拟 apo 构象时施加的随机平移和旋转幅度

In [ ]:
# ============================================================
# 超参数定义
# ============================================================
ATOM_FEAT_DIM = 10          # 配体原子特征维度 (与 toy_ign 一致)
RES_FEAT_DIM = 21           # 残基特征维度 (20 种标准氨基酸 + 1 other)
HIDDEN_DIM = 128            # 隐层维度
N_CROSS_ATTN = 4            # 交叉注意力层数
N_EPOCHS = 200              # 训练轮数
LR = 1e-3                   # 学习率
BATCH_SIZE = 1              # 变长结构，逐样本处理
SEED = 42
PERTURB_TRANS_STD = 2.0     # 扰动平移标准差 (Angstrom)
PERTURB_ROT_STD = 0.1       # 扰动旋转标准差 (弧度)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# 展示超参数
params_df = pd.DataFrame({
    '超参数': ['ATOM_FEAT_DIM', 'RES_FEAT_DIM', 'HIDDEN_DIM', 'N_CROSS_ATTN',
              'N_EPOCHS', 'LR', 'BATCH_SIZE', 'PERTURB_TRANS_STD', 'PERTURB_ROT_STD', 'DEVICE'],
    '值': [ATOM_FEAT_DIM, RES_FEAT_DIM, HIDDEN_DIM, N_CROSS_ATTN,
           N_EPOCHS, LR, BATCH_SIZE, f'{PERTURB_TRANS_STD} A', f'{PERTURB_ROT_STD} rad', str(DEVICE)],
    '说明': ['配体原子特征维度', '残基特征维度 (20 AA + 1 other)', '隐层维度',
            '交叉注意力层数', '训练轮数', '学习率', '批大小 (变长结构逐样本)',
            '扰动平移标准差', '扰动旋转标准差', '计算设备']
})
display(params_df)

## 2. 数据加载与特征提取

CarsiInduce 需要以下输入数据:

| 数据 | 来源 | 形状 | 说明 |
|------|------|------|------|
| 残基特征 | 口袋 PDB | (N_res, 21) | 20 种标准氨基酸 one-hot + 1 other |
| Holo CA 坐标 | 口袋 PDB | (N_res, 3) | 真实结合构象 (训练标签) |
| 扰动 CA 坐标 | 随机扰动生成 | (N_res, 3) | 模拟 apo 构象 (模型输入) |
| 配体原子特征 | 配体 mol2/sdf | (N_l, 10) | 元素 one-hot + 芳香性 |
| 配体原子坐标 | 配体 mol2/sdf | (N_l, 3) | 配体 3D 坐标 |

扰动过程: 对每个残基的 CA 坐标独立施加随机平移 (sigma=2.0 A) 和小角度旋转 (sigma=0.1 rad)，模拟从 holo 到 apo 的构象变化。

In [ ]:
# ============================================================
# 特征提取函数
# ============================================================

# ---- 20 种标准氨基酸 ----
STANDARD_AAS = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
                'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']

# ---- 配体元素 -> one-hot 映射 (9类 + 1 芳香性 = 10维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br']  # 8 种常见元素 + 1 other


def residue_features(resname):
    """
    构建 21 维残基特征向量：
      - 前 20 维: 氨基酸类型 one-hot (20 种标准氨基酸)
      - 第 21 维: 非标准氨基酸 (other)
    """
    feat = np.zeros(RES_FEAT_DIM, dtype=np.float32)
    resname = resname.strip().upper()
    if resname in STANDARD_AAS:
        feat[STANDARD_AAS.index(resname)] = 1.0
    else:
        feat[20] = 1.0  # other 类别
    return feat


def atom_features(atom):
    """
    构建 10 维原子特征向量 (用于配体原子):
      - 前 9 维: 元素类型 one-hot (C, N, O, S, F, P, Cl, Br, other)
      - 第 10 维: 是否为芳香性原子
    """
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(symbol)] = 1.0
    else:
        feat[8] = 1.0       # other 类别
    feat[9] = float(atom.GetIsAromatic())
    return feat


def parse_coreset(path):
    """解析 CoreSet.dat，返回 pdbid 列表 (此模型不需要 logKa 标签)。"""
    pdbids = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            pdbids.append(parts[0])
    print(f"[INFO] 从 CoreSet.dat 读取到 {len(pdbids)} 个复合物")
    return pdbids


def axis_angle_to_matrix(axis_angle):
    """
    将轴角表示 (3,) 转换为旋转矩阵 (3,3)，使用 Rodrigues 公式。

    axis_angle: (3,) tensor，方向为旋转轴，模长为旋转角度 (弧度)。
    返回: (3,3) 旋转矩阵。

    Rodrigues 公式: R = I + sin(theta)*K + (1 - cos(theta))*K^2
    其中 K 是旋转轴的反对称矩阵。
    """
    theta = torch.norm(axis_angle, dim=-1, keepdim=True).clamp(min=1e-8)
    k = axis_angle / theta                          # (3,) 归一化旋转轴
    # 构建反对称矩阵 K
    K = torch.zeros(3, 3, device=axis_angle.device, dtype=axis_angle.dtype)
    K[0, 1] = -k[2]; K[0, 2] = k[1]
    K[1, 0] = k[2];  K[1, 2] = -k[0]
    K[2, 0] = -k[1]; K[2, 1] = k[0]
    theta_scalar = theta.squeeze()
    R = (torch.eye(3, device=axis_angle.device, dtype=axis_angle.dtype)
         + torch.sin(theta_scalar) * K
         + (1 - torch.cos(theta_scalar)) * (K @ K))
    return R


def perturb_residue_coords(coords_np, rng):
    """
    对残基坐标施加随机扰动，模拟 apo 构象。

    对每个残基独立施加:
      - 随机平移 (各轴独立, sigma = PERTURB_TRANS_STD Angstrom)
      - 围绕质心的小随机旋转 (sigma = PERTURB_ROT_STD 弧度)

    参数:
      coords_np: (N_res, 3) numpy array, holo 残基 CA 坐标
      rng: numpy RandomState 对象
    返回:
      perturbed_coords: (N_res, 3) numpy array, 扰动后坐标
    """
    n_res = coords_np.shape[0]
    translations = rng.randn(n_res, 3).astype(np.float32) * PERTURB_TRANS_STD
    rotations = rng.randn(n_res, 3).astype(np.float32) * PERTURB_ROT_STD

    centroid = coords_np.mean(axis=0)
    perturbed = np.zeros_like(coords_np)
    for i in range(n_res):
        # 旋转: 以整体质心为中心，使用 Rodrigues 公式
        theta = np.linalg.norm(rotations[i])
        if theta > 1e-8:
            k = rotations[i] / theta
            K = np.array([[0, -k[2], k[1]],
                          [k[2], 0, -k[0]],
                          [-k[1], k[0], 0]], dtype=np.float32)
            R = (np.eye(3, dtype=np.float32)
                 + np.sin(theta) * K
                 + (1 - np.cos(theta)) * (K @ K))
            delta = coords_np[i] - centroid
            perturbed[i] = centroid + R @ delta
        else:
            perturbed[i] = coords_np[i].copy()
        # 平移
        perturbed[i] += translations[i]

    return perturbed


def extract_pocket_residues(pocket_path):
    """
    从口袋 PDB 文件中提取残基级信息。

    使用 BioPython PDBParser 解析 PDB，对每个残基提取:
      - CA 原子坐标 (3,)
      - 残基类型 one-hot (21,)

    返回:
      res_feats: (N_res, 21) numpy array
      res_coords: (N_res, 3) numpy array (CA 坐标)
    如果解析失败或无 CA 原子则返回 None。
    """
    parser = PDBParser(QUIET=True)
    try:
        structure = parser.get_structure('pocket', pocket_path)
    except Exception:
        return None

    res_feats_list = []
    res_coords_list = []

    for model in structure:
        for chain in model:
            for residue in chain:
                # 跳过水分子和杂原子 (HETATM)
                het_flag = residue.get_id()[0]
                if het_flag != ' ':
                    continue
                # 提取 CA 原子坐标
                if 'CA' not in residue:
                    continue
                ca_coord = residue['CA'].get_vector().get_array().astype(np.float32)
                resname = residue.get_resname()
                feat = residue_features(resname)
                res_feats_list.append(feat)
                res_coords_list.append(ca_coord)
        break  # 只取第一个 model

    if len(res_feats_list) == 0:
        return None

    res_feats = np.array(res_feats_list, dtype=np.float32)
    res_coords = np.array(res_coords_list, dtype=np.float32)
    return res_feats, res_coords


def load_ligand(pdbid):
    """
    加载配体分子，提取原子特征和坐标。

    返回:
      lig_feats: (N_l, 10) numpy array
      lig_coords: (N_l, 3) numpy array
    如果解析失败则返回 None。
    """
    ligand_mol2 = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_ligand.mol2")
    ligand_sdf = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_ligand.sdf")

    # mol2 优先, sdf 备选
    lig_mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if lig_mol is None and os.path.exists(ligand_sdf):
        supplier = Chem.SDMolSupplier(ligand_sdf, sanitize=False)
        for m in supplier:
            if m is not None:
                lig_mol = m
                break
    if lig_mol is None:
        return None
    try:
        Chem.SanitizeMol(lig_mol)
    except Exception:
        pass
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass

    conf = lig_mol.GetConformer()
    coords = np.array([conf.GetAtomPosition(i) for i in range(lig_mol.GetNumAtoms())],
                       dtype=np.float32)
    feats = np.array([atom_features(a) for a in lig_mol.GetAtoms()], dtype=np.float32)

    if feats.shape[0] == 0:
        return None
    return feats, coords


def build_carsiinduce_data(pdbid, rng):
    """
    为单个蛋白-配体复合物构建 CarsiInduce 训练数据。

    返回:
      res_feats:       (N_res, 21) 残基特征
      holo_coords:     (N_res, 3)  真实 holo CA 坐标 (训练标签)
      perturbed_coords:(N_res, 3)  扰动后 CA 坐标 (模拟 apo, 模型输入)
      lig_feats:       (N_l, 10)   配体原子特征
      lig_coords:      (N_l, 3)    配体原子坐标
    如果解析失败则返回 None。
    """
    pocket_path = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_pocket.pdb")

    # ---- 1. 提取口袋残基 ----
    pocket_result = extract_pocket_residues(pocket_path)
    if pocket_result is None:
        return None
    res_feats, holo_coords = pocket_result

    # ---- 2. 加载配体 ----
    lig_result = load_ligand(pdbid)
    if lig_result is None:
        return None
    lig_feats, lig_coords = lig_result

    # ---- 3. 对 holo 坐标施加扰动，模拟 apo 构象 ----
    perturbed_coords = perturb_residue_coords(holo_coords, rng)

    return res_feats, holo_coords, perturbed_coords, lig_feats, lig_coords

In [ ]:
# ============================================================
# 加载数据并展示一个样本
# ============================================================
print("正在构建残基级构象数据...")
pdbids = parse_coreset(CORESET_FILE)

rng = np.random.RandomState(SEED)
all_data = []
failed = 0
for pdbid in sorted(pdbids):
    result = build_carsiinduce_data(pdbid, rng)
    if result is None:
        failed += 1
        continue
    all_data.append(result)

print(f"成功加载 {len(all_data)} 个复合物, 失败 {failed} 个")

# 展示第一个样本的形状信息
sample = all_data[0]
sample_info = pd.DataFrame({
    '数据': ['残基特征 (res_feats)', 'Holo CA 坐标 (holo_coords)',
            '扰动 CA 坐标 (perturbed_coords)', '配体原子特征 (lig_feats)',
            '配体原子坐标 (lig_coords)'],
    '形状': [str(sample[0].shape), str(sample[1].shape),
            str(sample[2].shape), str(sample[3].shape),
            str(sample[4].shape)],
    '说明': ['残基类型 one-hot', '真实结合构象 (标签)', '模拟 apo 构象 (输入)',
            '元素 one-hot + 芳香性', '配体 3D 坐标']
})
display(sample_info)

## 3. 数据集与数据加载器

由于每个蛋白-配体复合物的残基数和配体原子数不同 (变长结构)，使用 `batch_size=1` 逐样本处理。数据集按 80/20 随机划分为训练集和测试集。

In [ ]:
# ============================================================
# Dataset 与 DataLoader
# ============================================================

class CarsiInduceDataset(Dataset):
    """
    CarsiInduce 数据集。
    每个样本: 扰动残基坐标 + 残基特征 + 配体特征/坐标 + 真实 holo 坐标。
    """
    def __init__(self, data_list):
        # data_list: list of (res_feats, holo_coords, perturbed_coords, lig_feats, lig_coords)
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        res_f, holo_c, pert_c, lig_f, lig_c = self.data[idx]
        return (torch.FloatTensor(res_f),       # (N_res, 21)
                torch.FloatTensor(holo_c),       # (N_res, 3)
                torch.FloatTensor(pert_c),       # (N_res, 3)
                torch.FloatTensor(lig_f),        # (N_l, 10)
                torch.FloatTensor(lig_c))        # (N_l, 3)


# ---- 随机 80/20 划分训练/测试集 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_idx, test_idx = indices[:split], indices[split:]

train_data = [all_data[i] for i in train_idx]
test_data = [all_data[i] for i in test_idx]

print(f"训练集: {len(train_data)}, 测试集: {len(test_data)}")

train_loader = DataLoader(CarsiInduceDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(CarsiInduceDataset(test_data), batch_size=BATCH_SIZE, shuffle=False)

# 展示数据集划分信息
split_df = pd.DataFrame({
    '数据集': ['训练集', '测试集', '总计'],
    '样本数': [len(train_data), len(test_data), len(all_data)],
    '比例': [f'{len(train_data)/len(all_data)*100:.1f}%',
            f'{len(test_data)/len(all_data)*100:.1f}%', '100.0%']
})
display(split_df)

## 4. 模型架构

ToyCarsiInduce 的核心架构:

```
残基特征(21) + 扰动坐标(3) ---> res_embed ---> [残基嵌入 (N_res, hidden)]
                                                        |
                                                        | Query
                                                        v
配体特征(10) + 配体坐标(3) ---> lig_embed ---> [配体嵌入 (N_l, hidden)]
                                                        |
                                                Key/Value|
                                                        v
                                           [交叉注意力 x N_CROSS_ATTN 层]
                                                        |
                                                        v
                                              [校正后的残基表示]
                                               /              \\
                                              v                v
                                    translation_head      rotation_head
                                      (N_res, 3)           (N_res, 3)
                                              \\              /
                                               v            v
                                        [应用旋转 + 平移校正]
                                                |
                                                v
                                    corrected_coords (N_res, 3)
```

**交叉注意力机制**: 残基作为 Query，配体作为 Key/Value。每个残基通过注意力权重从配体原子中"读取"与其构象校正相关的信息。这正是 CarsiInduce 的核心 -- 配体信息指导蛋白质构象调整。

In [ ]:
# ============================================================
# 模型架构 -- ToyCarsiInduce
# ============================================================

class CrossAttentionLayer(nn.Module):
    """
    单头交叉注意力层。

    Query 来自蛋白残基，Key/Value 来自配体原子。
    通过注意力机制，让每个残基从配体中"读取"与其校正相关的信息。
    包含残差连接和 LayerNorm。
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.scale = hidden_dim ** 0.5

    def forward(self, query, key_value):
        """
        参数:
          query:     (N_res, hidden) 残基嵌入 (作为 Query)
          key_value: (N_l, hidden)   配体嵌入 (作为 Key 和 Value)
        返回:
          out: (N_res, hidden) 经过交叉注意力更新后的残基表示
        """
        Q = self.q_proj(query)          # (N_res, hidden)
        K = self.k_proj(key_value)      # (N_l, hidden)
        V = self.v_proj(key_value)      # (N_l, hidden)

        # 计算注意力权重: (N_res, N_l)
        attn_weights = torch.matmul(Q, K.transpose(0, 1)) / self.scale
        attn_weights = torch.softmax(attn_weights, dim=-1)

        # 加权聚合配体信息
        attn_out = torch.matmul(attn_weights, V)  # (N_res, hidden)
        out = self.out_proj(attn_out)

        # 残差连接 + LayerNorm
        out = self.norm(query + out)
        return out


class ToyCarsiInduce(nn.Module):
    """
    残基级构象校正 -- ligand info guides protein conformational adjustment via cross-attention.

    核心思想：
      - 将扰动后的残基坐标和残基特征嵌入到隐空间；
      - 将配体原子特征和坐标嵌入到同一隐空间；
      - 通过多层交叉注意力，让残基从配体原子中"读取"校正信号；
      - 最终预测每个残基的平移向量和旋转轴角，
        用于将扰动构象校正回接近 holo 的状态。

    与传统对接方法的区别：
      这里关注的不是配体的位姿，而是 **蛋白质构象**--
      模型学习 "配体如何诱导蛋白质从 apo 变为 holo"。
    """
    def __init__(self, res_dim=RES_FEAT_DIM, lig_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        # 1. 残基嵌入: 残基特征 (21) + 3D坐标 (3) -> 隐空间
        self.res_embed = nn.Linear(res_dim + 3, hidden_dim)

        # 2. 配体嵌入: 原子特征 (10) + 3D坐标 (3) -> 隐空间
        self.lig_embed = nn.Linear(lig_dim + 3, hidden_dim)

        # 3. 交叉注意力层: 残基 (Query) 关注 配体 (Key/Value)
        self.cross_attn_layers = nn.ModuleList([
            CrossAttentionLayer(hidden_dim) for _ in range(N_CROSS_ATTN)
        ])

        # 4. 校正头: 预测每个残基的平移 (3) 和旋转轴角 (3)
        self.translation_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 3)
        )
        self.rotation_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 3)
        )

    def forward(self, res_feats, perturbed_coords, lig_feats, lig_coords):
        """
        参数:
          res_feats:        (N_res, 21)  残基类型 one-hot
          perturbed_coords: (N_res, 3)   扰动后的 CA 坐标 (模拟 apo)
          lig_feats:        (N_l, 10)    配体原子特征
          lig_coords:       (N_l, 3)     配体原子坐标

        返回:
          corrected_coords: (N_res, 3) 校正后的残基坐标 (预测 holo)
        """
        # Step 1: 嵌入残基 (特征 + 坐标 拼接)
        res_input = torch.cat([res_feats, perturbed_coords], dim=-1)  # (N_res, 24)
        res_h = self.res_embed(res_input)                              # (N_res, hidden)

        # Step 2: 嵌入配体 (特征 + 坐标 拼接)
        lig_input = torch.cat([lig_feats, lig_coords], dim=-1)        # (N_l, 13)
        lig_h = self.lig_embed(lig_input)                              # (N_l, hidden)

        # Step 3: 多层交叉注意力 -- 残基从配体中获取校正信号
        for cross_attn in self.cross_attn_layers:
            res_h = cross_attn(res_h, lig_h)

        # Step 4: 预测每个残基的平移和旋转校正
        pred_trans = self.translation_head(res_h)    # (N_res, 3)
        pred_rot = self.rotation_head(res_h)         # (N_res, 3) 轴角表示

        # Step 5: 应用校正 -- 先旋转后平移
        #   以扰动构象的质心为旋转中心
        centroid = perturbed_coords.mean(dim=0, keepdim=True)  # (1, 3)
        corrected = torch.zeros_like(perturbed_coords)

        for i in range(perturbed_coords.shape[0]):
            # 旋转: 围绕质心旋转
            R = axis_angle_to_matrix(pred_rot[i])                      # (3, 3)
            delta = perturbed_coords[i] - centroid.squeeze(0)          # (3,)
            rotated = centroid.squeeze(0) + R @ delta                  # (3,)
            # 平移
            corrected[i] = rotated + pred_trans[i]

        return corrected

In [ ]:
# ============================================================
# 模型实例化与参数统计
# ============================================================
model = ToyCarsiInduce(res_dim=RES_FEAT_DIM, lig_dim=ATOM_FEAT_DIM,
                       hidden_dim=HIDDEN_DIM).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"模型参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")
print()

# 按模块统计参数
module_params = []
for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    module_params.append({'模块': name, '参数量': f'{n_params:,}'})
display(pd.DataFrame(module_params))

print()
print(model)

## 5. 训练

训练目标: 最小化预测构象与真实 holo 构象之间的 RMSD (Root Mean Square Deviation)。

$$\text{RMSD} = \sqrt{\frac{1}{N_{res}} \sum_{i=1}^{N_{res}} \|\hat{\mathbf{x}}_i - \mathbf{x}_i^{holo}\|^2}$$

每 20 轮在测试集上评估，对比校正前 (扰动构象) 和校正后 (模型预测) 的 RMSD。

In [ ]:
# ============================================================
# 训练循环
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def rmsd_loss(pred_coords, true_coords):
    """
    计算预测坐标与真实坐标之间的 RMSD (Root Mean Square Deviation)。
    RMSD = sqrt(mean(||pred_i - true_i||^2))

    这是蛋白质构象预测中最常用的评估指标。
    """
    diff = pred_coords - true_coords                          # (N_res, 3)
    return torch.sqrt((diff ** 2).sum(dim=-1).mean())         # 标量


print(f"开始训练 {N_EPOCHS} 轮...\n")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_losses = []

    for res_f, holo_c, pert_c, lig_f, lig_c in train_loader:
        # 每个 batch 只有 1 个样本 (变长结构)，去掉 batch 维度
        res_f = res_f.squeeze(0).to(DEVICE)     # (N_res, 21)
        holo_c = holo_c.squeeze(0).to(DEVICE)   # (N_res, 3)
        pert_c = pert_c.squeeze(0).to(DEVICE)   # (N_res, 3)
        lig_f = lig_f.squeeze(0).to(DEVICE)     # (N_l, 10)
        lig_c = lig_c.squeeze(0).to(DEVICE)     # (N_l, 3)

        # 跳过残基数或配体原子数为 0 的样本
        if res_f.shape[0] == 0 or lig_f.shape[0] == 0:
            continue

        optimizer.zero_grad()
        corrected = model(res_f, pert_c, lig_f, lig_c)
        loss = rmsd_loss(corrected, holo_c)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # ---- 每 20 轮评估一次 ----
    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        val_rmsds_before = []
        val_rmsds_after = []
        with torch.no_grad():
            for res_f, holo_c, pert_c, lig_f, lig_c in test_loader:
                res_f = res_f.squeeze(0).to(DEVICE)
                holo_c = holo_c.squeeze(0).to(DEVICE)
                pert_c = pert_c.squeeze(0).to(DEVICE)
                lig_f = lig_f.squeeze(0).to(DEVICE)
                lig_c = lig_c.squeeze(0).to(DEVICE)
                if res_f.shape[0] == 0 or lig_f.shape[0] == 0:
                    continue

                corrected = model(res_f, pert_c, lig_f, lig_c)
                rmsd_before = rmsd_loss(pert_c, holo_c).item()
                rmsd_after = rmsd_loss(corrected, holo_c).item()
                val_rmsds_before.append(rmsd_before)
                val_rmsds_after.append(rmsd_after)

        avg_train = np.mean(train_losses) if train_losses else float('nan')
        avg_before = np.mean(val_rmsds_before) if val_rmsds_before else float('nan')
        avg_after = np.mean(val_rmsds_after) if val_rmsds_after else float('nan')
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  Train RMSD: {avg_train:.4f}  "
              f"|  Val Before: {avg_before:.4f}  |  Val After: {avg_after:.4f}")

## 6. 评估与可视化

在测试集上评估模型性能，主要关注:
- **校正前 RMSD**: 扰动构象 (模拟 apo) 与真实 holo 构象的偏差
- **校正后 RMSD**: 模型预测构象与真实 holo 构象的偏差
- **改善比例**: 校正后 RMSD 小于校正前的样本占比
- **RMSD < 2A / 5A 成功率**: 常用的构象预测质量阈值

In [ ]:
# ============================================================
# 测试集评估
# ============================================================
print("在测试集上评估...\n")

model.eval()
rmsds_before = []
rmsds_after = []

with torch.no_grad():
    for res_f, holo_c, pert_c, lig_f, lig_c in test_loader:
        res_f = res_f.squeeze(0).to(DEVICE)
        holo_c = holo_c.squeeze(0).to(DEVICE)
        pert_c = pert_c.squeeze(0).to(DEVICE)
        lig_f = lig_f.squeeze(0).to(DEVICE)
        lig_c = lig_c.squeeze(0).to(DEVICE)
        if res_f.shape[0] == 0 or lig_f.shape[0] == 0:
            continue

        corrected = model(res_f, pert_c, lig_f, lig_c)
        rmsd_b = rmsd_loss(pert_c, holo_c).item()
        rmsd_a = rmsd_loss(corrected, holo_c).item()
        rmsds_before.append(rmsd_b)
        rmsds_after.append(rmsd_a)

rmsds_before = np.array(rmsds_before)
rmsds_after = np.array(rmsds_after)

# ---- 计算统计指标 ----
mean_before = rmsds_before.mean()
mean_after = rmsds_after.mean()
median_before = np.median(rmsds_before)
median_after = np.median(rmsds_after)
improved = (rmsds_after < rmsds_before).sum()
total = len(rmsds_before)
pct_under_2 = np.mean(rmsds_after < 2.0) * 100   # RMSD < 2A 成功率
pct_under_5 = np.mean(rmsds_after < 5.0) * 100   # RMSD < 5A 成功率

# 用 DataFrame 展示结果
results_df = pd.DataFrame({
    '指标': ['样本数', '校正前 Mean RMSD', '校正后 Mean RMSD',
            '校正前 Median RMSD', '校正后 Median RMSD',
            '改善比例', 'RMSD < 2A', 'RMSD < 5A'],
    '值': [f'{total}',
          f'{mean_before:.4f} A', f'{mean_after:.4f} A',
          f'{median_before:.4f} A', f'{median_after:.4f} A',
          f'{improved}/{total} ({100*improved/total:.1f}%)',
          f'{pct_under_2:.1f}%', f'{pct_under_5:.1f}%']
})
display(results_df)

In [ ]:
# ============================================================
# 可视化: 校正前后 RMSD 对比
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# 左图: 散点图 (每个复合物的 RMSD before vs after)
ax = axes[0]
ax.scatter(rmsds_before, rmsds_after, alpha=0.6, edgecolors='k', linewidths=0.5, s=40,
           c='steelblue', label='Complexes')
# 对角线 y = x (以上 = 校正后更差，以下 = 校正后更好)
lo = 0
hi = max(rmsds_before.max(), rmsds_after.max()) + 0.5
ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.0, label='y = x (no change)')
ax.set_xlabel('RMSD Before Correction (A)', fontsize=12)
ax.set_ylabel('RMSD After Correction (A)', fontsize=12)
ax.set_title(f'CarsiInduce: Per-complex RMSD\n'
             f'Improved: {improved}/{total} ({100*improved/total:.1f}%)', fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect('equal')

# 右图: 箱线图对比
ax = axes[1]
bp = ax.boxplot([rmsds_before, rmsds_after],
                labels=['Before\n(Perturbed)', 'After\n(Corrected)'],
                patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#FF9999')
bp['boxes'][1].set_facecolor('#99CCFF')
ax.set_ylabel('RMSD to Holo (A)', fontsize=12)
ax.set_title(f'CarsiInduce: RMSD Distribution\n'
             f'Mean: {mean_before:.2f}A -> {mean_after:.2f}A', fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 总结

本教程展示了 CarsiInduce 的核心思想 -- **配体诱导蛋白质构象变化 (induced-fit docking)**:

1. **问题定义**: 与传统对接方法预测配体位姿不同，CarsiInduce 关注蛋白质构象的变化。模型学习"配体如何诱导蛋白质从 apo (未结合) 构象变为 holo (结合) 构象"。

2. **残基级表示**: 使用 CA 原子坐标 + 残基类型 one-hot 表示蛋白质口袋中的每个残基，配体使用原子特征 + 3D 坐标表示。

3. **交叉注意力机制**: 这是 CarsiInduce 的核心。残基作为 Query，配体原子作为 Key/Value，通过多层交叉注意力让配体信息引导蛋白质残基的构象校正。

4. **刚体变换校正**: 模型为每个残基预测平移向量 (3D) 和旋转轴角 (3D)，使用 Rodrigues 公式将旋转应用到扰动构象上。

5. **RMSD 评估**: 使用 RMSD (Root Mean Square Deviation) 评估预测构象与真实 holo 构象的偏差，是蛋白质构象预测的标准指标。

**注意**: 本教程为简化版教学演示，使用 CASF-2016 核心集的 285 个复合物进行训练和测试。实际 CarsiInduce 模型使用更大规模的数据集，并包含更复杂的特征提取和网络架构。